# Starbucks Rewards — measuring a zero-discount personalisation campaign

**Independent case study · August 2026**

On 10 March 2026 Starbucks relaunched Rewards with three tiers. Buried in the
program terms is a mechanic that applies **only to the entry tier**: Green
members' Stars expire six months after the month they were earned, but can be
extended by one month — indefinitely — by a single qualifying action. Gold and
Reserve Stars never expire.

That mechanic is the most forgiving expiry rule in the category, and it is
invisible unless you read the terms. This notebook works through what it would
take to test whether telling low-frequency Green members about it produces
**profitable incremental visits**.

> Nothing here is Starbucks performance data. Public facts are sourced;
> everything else is labelled `CASE_ASSUMPTION`, `PROPOSED_TARGET` or
> `CALCULATED`. The pilot data is simulated to demonstrate the analysis
> pipeline.

In [1]:
import sys; sys.path.insert(0, "../analysis")
import numpy as np, pandas as pd
pd.set_option("display.width", 120)

import assumptions, power, economics, simulate
from assumptions import PUBLIC, A, TARGETS

## 1. What is actually known

Every public number carries its source. The member count is the one to watch:
it was not restated in the Q3 FY2026 release, so the newest available figure is
already six months old.

In [2]:
for k, v in PUBLIC.items():
    shown = f"{v.value:,}" if isinstance(v.value, (int, float)) else str(v.value)
    print(f"{k:<45} {shown:>12}  [{v.label}]")
    if v.source: print(f"{'':<45} source: {v.source}")

us_90day_active_members_q1fy26                  35,500,000  [PUBLIC_FACT]
                                              source: https://investor.starbucks.com/news/financial-releases/news-details/2026/Starbucks-Unveils-Reimagined-Loyalty-Program-to-Deliver-More-Meaningful-Value-Personalization-and-Engagement-for-Members/default.aspx
rewards_share_us_company_operated_revenue_fy25          0.6  [PUBLIC_FACT]
                                              source: https://investor.starbucks.com/news/financial-releases/news-details/2026/Starbucks-Is-Back-Turning-Momentum-Into-Long-Term-Sustainable-Growth/default.aspx
us_comp_sales_q3fy26                                 0.079  [PUBLIC_FACT]
                                              source: https://investor.starbucks.com/news/financial-releases/news-details/2026/Starbucks-Reports-Q3-Fiscal-Year-2026-Results/default.aspx
us_comp_transactions_q3fy26                          0.042  [PUBLIC_FACT]
                                              s

## 2. The business question

Growth and the relaunch occupy the same window: U.S. comparable sales rose 7.9%
in Q3 FY2026, driven by a 4.2% rise in transactions. Menu, pricing, throughput,
seasonality and the loyalty change all moved together.

Without a holdout, none of that lift can be attributed to the program — and an
attribution error compounds every time the company scales what it believes
worked.

> **Five months after the relaunch, how much of the change in low-frequency
> Green member behaviour is genuinely incremental — and can it be grown
> profitably without further discounting?**

## 3. Derived quantities

Everything downstream rests on these. They are derived, not observed.

In [3]:
lo, hi = assumptions.star_value_range()
print(f"baseline transactions / member / 12 weeks : {assumptions.baseline_txn_per_window():.2f}")
print(f"contribution margin per transaction      : ${assumptions.contribution_margin_per_txn():.2f}")
print(f"Star value implied by published ladder   : ${lo:.3f} - ${hi:.3f}")
print(f"alpha after Bonferroni (2 comparisons)   : {assumptions.corrected_alpha():.3f}")

baseline transactions / member / 12 weeks : 4.20
contribution margin per transaction      : $2.12
Star value implied by published ladder   : $0.033 - $0.060
alpha after Bonferroni (2 comparisons)   : 0.025


## 4. Sample size

The outcome is a count and an over-dispersed one, so variance is modelled as
negative binomial. Dispersion `k` cannot be known without internal data, so
every result is reported across a range.

In [4]:
print(power.requirement_table().to_string(index=False))
print()
print(power.mde_table().to_string(index=False))

 dispersion_k   sd  n_per_arm  total_3_arms
            5 2.78       3332          9996
            3 3.17       4346         13038
            2 3.61       5613         16839
            1 4.67       9415         28245

 total_N  per_arm  mde_k5  mde_k3  mde_k2  mde_k1
   30000    10000    2.89    3.30    3.75    4.85
   60000    20000    2.04    2.33    2.65    3.43
  100000    33333    1.58    1.81    2.05    2.66
  150000    50000    1.29    1.47    1.68    2.17


**10,000 per arm — 30,000 total — is sufficient even at the most severe
dispersion tested.**

The original plan called for 100,000. That would detect a 2.7% lift: roughly
three times more power than the decision needs. Surplus power is not free —
every extra member sitting in the holdout is forgone revenue. Running the
calculation made the test 70% cheaper and made the holdout defensible to
whoever has to approve it.

## 5. Why not simply discount

The obvious alternative is a $2-off offer. The economics rule it out.

In [5]:
print(economics.discount_comparison().to_string(index=False))

activation_rate net_at_5%_lift net_at_10%_lift
             5%           $262          $4,725
            10%        −$3,937            $525
            20%       −$12,337         −$7,875


A discount campaign is profitable only when **few people use it**. The
marketing definition of success — high activation — is the financial definition
of failure.

That is the argument for a mechanic with no incremental incentive at all.

## 6. The cost that is not on the marketing budget

A zero-discount campaign looks free. It is not.

Under standard loyalty accounting, unredeemed Stars sit as deferred revenue and
are recognised as revenue when they expire — *breakage*. A campaign that stops
Stars from expiring converts recognised revenue back into an obligation to
serve.

How many Stars are exposed during a 12-week window is itself uncertain, so it
is parameterised rather than assumed.

In [6]:
print(f"Stars at risk, 'flow'  (one window of earning) : {economics.stars_at_risk('flow'):.0f}")
print(f"Stars at risk, 'stock' (full 6-month balance)  : {economics.stars_at_risk('stock'):.0f}")
print()
print(economics.breakeven_table().to_string(index=False))

Stars at risk, 'flow'  (one window of earning) : 36
Stars at risk, 'stock' (full 6-month balance)  : 76

breakage_reduction breakeven_lift_flow breakeven_lift_stock
                0%               1.01%                1.01%
                5%               2.01%                3.15%
               10%               3.01%                5.29%
               15%               4.01%                7.44%
               20%               5.01%                9.58%
               30%               7.01%               13.87%


Under the conservative `stock` pool, a campaign that rescues one in ten
otherwise-expiring Stars needs roughly a 5% lift to break even.

**That is why the target is ≈5% under the conservative `stock` assumption.**
It is not a round number picked for
the slide.

## 7. Simulated pilot

Two scenarios are run. The second is the one worth reading — a framework that
only demonstrates its own success is a brochure.

In [7]:
decisions = simulate.run_all()


=== A_personalisation_works ===
           arm     n  txn_per_member  activation_rate  optout_rate  lift_vs_holdout    ci_low  ci_high      p_value significant
     A_holdout 10000          4.2040           0.0000       0.0037              NaN       NaN      NaN          NaN        None
     B_generic 10000          4.2767           0.0635       0.0125         0.017293 -0.006930 0.041516 1.617376e-01       False
C_personalised 10000          4.5008           0.0999       0.0143         0.070599  0.045729 0.095469 2.639201e-08        True
  breakeven lift required : 5.72%
  net contribution        : +$1,194
  VERDICT                 : SCALE — incremental, profitable, and within guardrails

=== B_engagement_without_behaviour ===
           arm     n  txn_per_member  activation_rate  optout_rate  lift_vs_holdout    ci_low  ci_high  p_value significant
     A_holdout 10000          4.2040           0.0000       0.0037              NaN       NaN      NaN      NaN        None
     B_generic

In [8]:
decisions[["scenario","lift_vs_holdout","activation_rate",
           "net_contribution","verdict"]]

                         scenario  ...                                            verdict
0         A_personalisation_works  ...  SCALE — incremental, profitable, and within gu...
1  B_engagement_without_behaviour  ...  DO NOT SCALE — no incremental transaction effe...

[2 rows x 5 columns]

### Reading scenario B

Activation reaches 17% — a number that would headline any engagement dashboard
as a success. Transactions barely move, and once lost breakage is charged the
campaign destroys value.

**The decision rule catches it. A CTR dashboard would not.**

That gap is the entire argument for holdout-based measurement, and it is why
the primary metric is incremental contribution margin rather than any
engagement metric.

## 8. What I would ask Starbucks for

The single most load-bearing unknown in this whole model is the historical
breakage rate by frequency decile. Everything else is secondary.

1. Member-level transaction history and Star balances with expiry dates
2. **Historical breakage rate by frequency decile**
3. Product-level variable margin and true incentive cost
4. Offer exposure and activation events at member level
5. Assignment records for experiments already running, to avoid contamination
6. Opt-out and complaint data by campaign
7. The March 2026 tier-assignment thresholds — status was assigned from 2025
   behaviour, so a discontinuity exists that could be analysed without running
   any test at all